In [ ]:
# ==========================================================
# CELL 1 — IMPORTS
# ==========================================================

import os
import glob
import shutil
import json

import numpy as np
import pandas as pd

from scipy.stats import wilcoxon
from statsmodels.stats.contingency_tables import mcnemar

print("✓ Imports loaded")

In [ ]:
# ==========================================================
# CELL 2 — SAFE GOOGLE DRIVE CONNECTION
# ==========================================================

if os.path.exists("/content/drive/MyDrive"):
    print("Google Drive is already mounted.")

else:
    from google.colab import drive
    drive.mount("/content/drive")

    print("Google Drive mounted successfully.")


In [ ]:
# ==========================================================
# CELL 3 — CONFIRM PROJECT
# ==========================================================

PROJECT_DIR = (
    "/content/drive/MyDrive/"
    "RAG_ICH_MASTER"
)

if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(
        "STOP: RAG_ICH_MASTER was not found."
    )

print("✓ Project confirmed:")
print(PROJECT_DIR)

In [ ]:
# ==========================================================
# CELL 4 — FIND COMPLETED EVALUATION WORKBOOK
# READ ONLY
# ==========================================================

xlsx_files = glob.glob(
    os.path.join(
        PROJECT_DIR,
        "**",
        "*.xlsx"
    ),
    recursive=True
)

print(
    f"Found {len(xlsx_files)} Excel files.\n"
)

for i, path in enumerate(xlsx_files):
    print(f"[{i}] {path}")

In [ ]:
# ==========================================================
# CELL 5 — SET COMPLETED SCORED WORKBOOK
# ==========================================================

SCORED_FILE = r"/content/drive/MyDrive/RAG_ICH_MASTER/results/stage4_FINAL_HYBRID/FINAL_HYBRID_blinded_evaluation.xlsx"

if not os.path.isfile(SCORED_FILE):
    raise FileNotFoundError(
        f"STOP: File not found:\n{SCORED_FILE}"
    )

print("✓ Scored workbook found:")
print(SCORED_FILE)

In [ ]:
# ==========================================================
# CELL 6 — LOAD COMPLETED BLINDED EVALUATION
# ==========================================================

scored_df = pd.read_excel(
    SCORED_FILE
)

print("Rows:", len(scored_df))

print("\nColumns:")
for col in scored_df.columns:
    print("-", col)

display(
    scored_df.head()
)

In [ ]:
# ==========================================================
# CELL 7 — VALIDATE REQUIRED COLUMNS
# ==========================================================

required_columns = [
    "response_id",
    "question_id",
    "faithfulness_0_2",
    "clinical_correctness_0_2",
    "completeness_0_2",
    "conciseness_0_1",
    "unsupported_claims_yes_no"
]

missing_columns = [
    col
    for col in required_columns
    if col not in scored_df.columns
]

if missing_columns:

    raise ValueError(
        "STOP: Missing required columns:\n"
        + "\n".join(missing_columns)
    )

print("✓ Required columns present")

In [ ]:
# ==========================================================
# CELL 8 — STANDARDISE NUMERIC SCORES
# ==========================================================

numeric_score_cols = [
    "faithfulness_0_2",
    "clinical_correctness_0_2",
    "completeness_0_2",
    "conciseness_0_1"
]

for col in numeric_score_cols:

    scored_df[col] = pd.to_numeric(
        scored_df[col],
        errors="coerce"
    )


print("Missing numeric values:")

print(
    scored_df[
        numeric_score_cols
    ].isna().sum()
)

In [ ]:
# ==========================================================
# CELL 9 — FINAL SCORE VALIDATION
# ==========================================================

if len(scored_df) != 58:

    raise ValueError(
        f"STOP: Expected 58 responses, "
        f"but found {len(scored_df)}."
    )


for col in [
    "faithfulness_0_2",
    "clinical_correctness_0_2",
    "completeness_0_2"
]:

    invalid = scored_df.loc[
        ~scored_df[col].isin(
            [0, 1, 2]
        ),
        [
            "response_id",
            col
        ]
    ]

    if len(invalid) > 0:

        print(
            f"\nInvalid values in {col}:"
        )

        display(invalid)

        raise ValueError(
            f"STOP: Invalid values in {col}"
        )


invalid_conciseness = scored_df.loc[
    ~scored_df[
        "conciseness_0_1"
    ].isin(
        [0, 1]
    ),
    [
        "response_id",
        "conciseness_0_1"
    ]
]


if len(invalid_conciseness) > 0:

    display(
        invalid_conciseness
    )

    raise ValueError(
        "STOP: Invalid conciseness scores."
    )


print("✓ 58 responses")
print("✓ No missing numerical scores")
print("✓ All score ranges valid")

In [ ]:
# ==========================================================
# CELL 10 — VALIDATE UNSUPPORTED CLAIMS
# ==========================================================

scored_df[
    "unsupported_claims_yes_no"
] = (
    scored_df[
        "unsupported_claims_yes_no"
    ]
    .astype("string")
    .str.strip()
    .str.lower()
)


print(
    scored_df[
        "unsupported_claims_yes_no"
    ].value_counts(
        dropna=False
    )
)


invalid_unsupported = scored_df.loc[
    ~scored_df[
        "unsupported_claims_yes_no"
    ].isin(
        ["yes", "no"]
    ),
    [
        "response_id",
        "unsupported_claims_yes_no"
    ]
]


if len(invalid_unsupported) > 0:

    print(
        "\nInvalid unsupported-claim values:"
    )

    display(
        invalid_unsupported
    )

    raise ValueError(
        "STOP: Correct unsupported-claim values "
        "before unblinding."
    )


print(
    "\n✓ Unsupported claims column valid"
)

In [ ]:
# ==========================================================
# CELL 11 — CHECK RESPONSE IDs
# ==========================================================

if scored_df[
    "response_id"
].isna().any():

    raise ValueError(
        "STOP: Missing response_id."
    )


if scored_df[
    "response_id"
].duplicated().any():

    duplicate_ids = scored_df.loc[
        scored_df[
            "response_id"
        ].duplicated(
            keep=False
        ),
        "response_id"
    ]

    print(
        duplicate_ids
    )

    raise ValueError(
        "STOP: Duplicate response_id values found."
    )


print(
    "✓ 58 unique response IDs"
)

In [ ]:
# ==========================================================
# CELL 12 — ANALYSIS OUTPUT DIRECTORY
# ==========================================================

ANALYSIS_DIR = os.path.join(
    PROJECT_DIR,
    "results",
    "stage4_FINAL_HYBRID_ANALYSIS"
)

os.makedirs(
    ANALYSIS_DIR,
    exist_ok=True
)

print("Analysis directory:")
print(ANALYSIS_DIR)

In [ ]:
# ==========================================================
# CELL 13 — FREEZE BLINDED SCORES
# ==========================================================

FROZEN_FILE = os.path.join(
    ANALYSIS_DIR,
    "FINAL_HYBRID_blinded_scores_FROZEN.xlsx"
)


if os.path.exists(FROZEN_FILE):

    print(
        "Frozen file already exists."
    )

    print(
        "It will NOT be overwritten:"
    )

    print(
        FROZEN_FILE
    )


else:

    scored_df.to_excel(
        FROZEN_FILE,
        index=False
    )

    print(
        "✓ Frozen blinded evaluation saved:"
    )

    print(
        FROZEN_FILE
    )

In [ ]:
# ==========================================================
# CELL 14 — FIND UNBLINDING KEY
# READ ONLY
# ==========================================================

key_matches = glob.glob(
    os.path.join(
        PROJECT_DIR,
        "**",
        "*unblind*.csv"
    ),
    recursive=True
)


print(
    f"Found {len(key_matches)} "
    "possible unblinding keys:\n"
)


for i, path in enumerate(
    key_matches
):

    print(
        f"[{i}] {path}"
    )

In [ ]:
# ==========================================================
# CELL 15 — SET FINAL UNBLINDING KEY
# ==========================================================

UNBLINDING_FILE = r"/content/drive/MyDrive/RAG_ICH_MASTER/results/stage4_FINAL_HYBRID/FINAL_unblinding_key.csv"


if not os.path.isfile(
    UNBLINDING_FILE
):

    raise FileNotFoundError(
        f"STOP: Unblinding key not found:\n"
        f"{UNBLINDING_FILE}"
    )


print(
    "✓ Unblinding key found:"
)

print(
    UNBLINDING_FILE
)

In [ ]:
# ==========================================================
# CELL 16 — LOAD UNBLINDING KEY
# ==========================================================

key_df = pd.read_csv(
    UNBLINDING_FILE
)


print(
    "Key rows:",
    len(key_df)
)

print(
    "\nKey columns:"
)

print(
    key_df.columns.tolist()
)


display(
    key_df.head()
)

In [ ]:
# ==========================================================
# CELL 17 — VALIDATE UNBLINDING KEY
# ==========================================================

required_key_cols = [
    "response_id",
    "question_id",
    "condition"
]


missing_key_cols = [
    col
    for col in required_key_cols
    if col not in key_df.columns
]


if missing_key_cols:

    raise ValueError(
        "STOP: Missing key columns:\n"
        + "\n".join(
            missing_key_cols
        )
    )


if len(key_df) != 58:

    raise ValueError(
        f"STOP: Expected 58 rows in key, "
        f"found {len(key_df)}."
    )


if key_df[
    "response_id"
].duplicated().any():

    raise ValueError(
        "STOP: Duplicate response IDs "
        "in unblinding key."
    )


print(
    key_df[
        "condition"
    ].value_counts(
        dropna=False
    )
)

In [ ]:
# ==========================================================
# CELL 18 — UNBLIND FINAL STAGE 4
# ==========================================================

final_df = scored_df.merge(

    key_df[
        [
            "response_id",
            "question_id",
            "condition"
        ]
    ],

    on=[
        "response_id",
        "question_id"
    ],

    how="left",

    validate="one_to_one"
)


print(
    final_df[
        "condition"
    ].value_counts(
        dropna=False
    )
)

In [ ]:
# ==========================================================
# DIAGNOSE THE 2 UNMATCHED RESPONSES
# ==========================================================

failed = final_df.loc[
    final_df["condition"].isna(),
    [
        "response_id",
        "question_id"
    ]
].copy()

print("Unmatched responses:")
display(failed)


debug = failed.merge(
    key_df[
        [
            "response_id",
            "question_id",
            "condition"
        ]
    ],
    on="response_id",
    how="left",
    suffixes=(
        "_scored",
        "_key"
    )
)

print("\nCompare scored workbook vs unblinding key:")
display(debug)


In [ ]:
# ==========================================================
# CHECK THE TWO AFFECTED ROWS
# ==========================================================

check_ids = ["S042", "S045"]

cols_to_check = [
    "response_id",
    "question_id",
    "faithfulness_0_2",
    "clinical_correctness_0_2",
    "completeness_0_2",
    "conciseness_0_1",
    "unsupported_claims_yes_no"
]

display(
    scored_df.loc[
        scored_df["response_id"].isin(check_ids),
        cols_to_check
    ]
)

In [ ]:
# ==========================================================
# CHECK ALL QUESTION IDs AGAINST UNBLINDING KEY
# ==========================================================

id_check = scored_df[
    [
        "response_id",
        "question_id"
    ]
].merge(
    key_df[
        [
            "response_id",
            "question_id"
        ]
    ].rename(
        columns={
            "question_id": "question_id_key"
        }
    ),
    on="response_id",
    how="left",
    validate="one_to_one"
)

id_check["match"] = (
    id_check["question_id"].astype(str).str.strip()
    ==
    id_check["question_id_key"].astype(str).str.strip()
)

mismatches = id_check.loc[
    ~id_check["match"]
]

print(
    "Question-ID mismatches:",
    len(mismatches)
)

display(mismatches)

In [ ]:
# ==========================================================
# CELL 18 — UNBLIND USING RESPONSE_ID
# AND RESTORE AUTHORITATIVE QUESTION_ID
# ==========================================================

# Preserve the scored workbook's original question_id
# for audit purposes.
analysis_df = scored_df.copy()

analysis_df = analysis_df.rename(
    columns={
        "question_id":
            "question_id_original_scored"
    }
)


# Merge using response_id only.
# response_id is the unique blinded-response identifier.
final_df = analysis_df.merge(
    key_df[
        [
            "response_id",
            "question_id",
            "condition"
        ]
    ],
    on="response_id",
    how="left",
    validate="one_to_one"
)


print(
    final_df["condition"]
    .value_counts(dropna=False)
)


print("\nCorrected rows:")

display(
    final_df.loc[
        final_df["response_id"].isin(
            ["S042", "S045"]
        ),
        [
            "response_id",
            "question_id_original_scored",
            "question_id",
            "condition"
        ]
    ]
)

In [ ]:
# ==========================================================
# FINAL UNBLINDING INTEGRITY CHECK
# ==========================================================

assert len(final_df) == 58

assert final_df["response_id"].nunique() == 58

assert final_df["condition"].notna().all()

assert (
    final_df["condition"]
    .value_counts()
    .get("RAG", 0)
    == 29
)

assert (
    final_df["condition"]
    .value_counts()
    .get("Baseline", 0)
    == 29
)

assert (
    final_df.groupby("question_id")
    .size()
    .eq(2)
    .all()
)

print("✓ 58 unique responses")
print("✓ RAG = 29")
print("✓ Baseline = 29")
print("✓ Every question has exactly 2 responses")
print("✓ Unblinding dataset is structurally valid")

In [ ]:
# ==========================================================
# CELL 19 — VALIDATE UNBLINDING
# ==========================================================

if final_df[
    "condition"
].isna().any():

    failed = final_df.loc[
        final_df[
            "condition"
        ].isna(),
        [
            "response_id",
            "question_id"
        ]
    ]

    display(
        failed
    )

    raise ValueError(
        "STOP: Some responses failed to unblind."
    )


condition_counts = (
    final_df[
        "condition"
    ].value_counts()
)


if (
    condition_counts.get(
        "RAG",
        0
    ) != 29
):

    raise ValueError(
        "STOP: Expected 29 RAG responses."
    )


if (
    condition_counts.get(
        "Baseline",
        0
    ) != 29
):

    raise ValueError(
        "STOP: Expected 29 baseline responses."
    )


print(
    "✓ RAG = 29"
)

print(
    "✓ Baseline = 29"
)

print(
    "✓ Unblinding successful"
)

In [ ]:
# ==========================================================
# CELL 20 — TOTAL QUALITY SCORE
# ==========================================================

final_df[
    "total_score_0_7"
] = (

    final_df[
        "faithfulness_0_2"
    ]

    +

    final_df[
        "clinical_correctness_0_2"
    ]

    +

    final_df[
        "completeness_0_2"
    ]

    +

    final_df[
        "conciseness_0_1"
    ]
)


if not final_df[
    "total_score_0_7"
].between(
    0,
    7
).all():

    raise ValueError(
        "STOP: Total score outside 0–7."
    )


print(
    final_df[
        "total_score_0_7"
    ].describe()
)


print(
    "\n✓ All total scores valid"
)

In [ ]:
# ==========================================================
# CELL 21 — DESCRIPTIVE STATISTICS
# ==========================================================

outcomes = [
    "faithfulness_0_2",
    "clinical_correctness_0_2",
    "completeness_0_2",
    "conciseness_0_1",
    "total_score_0_7"
]


def q1(x):
    return x.quantile(
        0.25
    )


def q3(x):
    return x.quantile(
        0.75
    )


summary_table = (
    final_df
    .groupby(
        "condition"
    )[outcomes]
    .agg(
        [
            "mean",
            "std",
            "median",
            q1,
            q3,
            "min",
            "max"
        ]
    )
    .round(
        3
    )
)


display(
    summary_table
)

In [ ]:
# ==========================================================
# CELL 22 — THESIS-FRIENDLY DESCRIPTIVE TABLE
# ==========================================================

rows = []


for outcome in outcomes:

    for condition in [
        "RAG",
        "Baseline"
    ]:

        x = final_df.loc[
            final_df[
                "condition"
            ] == condition,
            outcome
        ]

        rows.append(
            {
                "outcome":
                    outcome,

                "condition":
                    condition,

                "n":
                    len(x),

                "mean":
                    x.mean(),

                "sd":
                    x.std(),

                "median":
                    x.median(),

                "q1":
                    x.quantile(
                        0.25
                    ),

                "q3":
                    x.quantile(
                        0.75
                    )
            }
        )


descriptive_long = pd.DataFrame(
    rows
).round(
    3
)


display(
    descriptive_long
)

In [ ]:
# ==========================================================
# CELL 23 — UNSUPPORTED CLINICAL CLAIMS
# ==========================================================

final_df[
    "has_unsupported_claim"
] = (

    final_df[
        "unsupported_claims_yes_no"
    ]
    .eq(
        "yes"
    )
)


unsupported_summary = (
    final_df
    .groupby(
        "condition"
    )[
        "has_unsupported_claim"
    ]
    .agg(
        unsupported_count="sum",
        total="count",
        proportion="mean"
    )
)


unsupported_summary[
    "percentage"
] = (

    unsupported_summary[
        "proportion"
    ]

    * 100
)


display(
    unsupported_summary[
        [
            "unsupported_count",
            "total",
            "percentage"
        ]
    ].round(
        2
    )
)

In [ ]:
# ==========================================================
# CELL 24 — PAIRED TOTAL-SCORE DATA
# ==========================================================

paired_total = final_df.pivot(
    index="question_id",
    columns="condition",
    values="total_score_0_7"
)


if (
    "RAG"
    not in paired_total.columns
    or
    "Baseline"
    not in paired_total.columns
):

    raise ValueError(
        "STOP: Could not create paired RAG/Baseline table."
    )


if paired_total[
    [
        "RAG",
        "Baseline"
    ]
].isna().any().any():

    raise ValueError(
        "STOP: One or more questions "
        "do not have both conditions."
    )


paired_total[
    "RAG_minus_Baseline"
] = (

    paired_total[
        "RAG"
    ]

    -

    paired_total[
        "Baseline"
    ]
)


display(
    paired_total
)

In [ ]:
# ==========================================================
# CELL 25 — RAG HIGHER / TIE / BASELINE HIGHER
# ==========================================================

difference = paired_total[
    "RAG_minus_Baseline"
]


rag_higher = (
    difference > 0
).sum()


ties = (
    difference == 0
).sum()


baseline_higher = (
    difference < 0
).sum()


print(
    "RAG higher:",
    rag_higher,
    "/ 29"
)

print(
    "Tie:",
    ties,
    "/ 29"
)

print(
    "Baseline higher:",
    baseline_higher,
    "/ 29"
)


print(
    "\nMean paired difference:",
    round(
        difference.mean(),
        3
    )
)


print(
    "Median paired difference:",
    round(
        difference.median(),
        3
    )
)

In [ ]:
# ==========================================================
# CELL 26 — PAIRED WILCOXON TESTS
# ==========================================================

wilcoxon_results = []


for outcome in outcomes:

    paired = final_df.pivot(
        index="question_id",
        columns="condition",
        values=outcome
    ).dropna()


    rag = paired[
        "RAG"
    ]


    baseline = paired[
        "Baseline"
    ]


    differences = (
        rag
        -
        baseline
    )


    try:

        statistic, p_value = wilcoxon(
            rag,
            baseline,
            alternative="two-sided",
            zero_method="wilcox"
        )

    except ValueError:

        statistic = np.nan
        p_value = np.nan


    wilcoxon_results.append(
        {
            "outcome":
                outcome,

            "n_pairs":
                len(paired),

            "RAG_median":
                rag.median(),

            "Baseline_median":
                baseline.median(),

            "median_difference":
                differences.median(),

            "W":
                statistic,

            "p_value":
                p_value
        }
    )


wilcoxon_df = pd.DataFrame(
    wilcoxon_results
)


display(
    wilcoxon_df
)

In [ ]:
# ==========================================================
# CELL 27 — FORMAT P VALUES
# ==========================================================

def format_p(p):

    if pd.isna(p):

        return "NA"

    if p < 0.001:

        return "<0.001"

    return f"{p:.3f}"


wilcoxon_df[
    "p_formatted"
] = (
    wilcoxon_df[
        "p_value"
    ].apply(
        format_p
    )
)


display(
    wilcoxon_df
)

In [ ]:
# ==========================================================
# CELL 28 — PAIR UNSUPPORTED CLAIMS
# ==========================================================

paired_unsupported = final_df.pivot(
    index="question_id",
    columns="condition",
    values="has_unsupported_claim"
)


if paired_unsupported[
    [
        "RAG",
        "Baseline"
    ]
].isna().any().any():

    raise ValueError(
        "STOP: Unsupported-claim pairing incomplete."
    )


paired_unsupported[
    "RAG"
] = (
    paired_unsupported[
        "RAG"
    ].astype(
        bool
    )
)


paired_unsupported[
    "Baseline"
] = (
    paired_unsupported[
        "Baseline"
    ].astype(
        bool
    )
)


display(
    paired_unsupported
)

In [ ]:
# ==========================================================
# CELL 29 — MCNEMAR 2×2 TABLE
# ==========================================================

mcnemar_table = pd.crosstab(
    paired_unsupported[
        "RAG"
    ],
    paired_unsupported[
        "Baseline"
    ]
)


mcnemar_table = (
    mcnemar_table
    .reindex(
        index=[
            False,
            True
        ],
        columns=[
            False,
            True
        ],
        fill_value=0
    )
)


print(
    "Rows = RAG unsupported claim"
)

print(
    "Columns = Baseline unsupported claim\n"
)


display(
    mcnemar_table
)

In [ ]:
# ==========================================================
# CELL 30 — EXACT MCNEMAR TEST
# ==========================================================

result = mcnemar(
    mcnemar_table.values,
    exact=True
)


print(
    "McNemar statistic:",
    result.statistic
)


print(
    "Exact p-value:",
    result.pvalue
)


print(
    "Formatted p-value:",
    format_p(
        result.pvalue
    )
)

In [ ]:
# ==========================================================
# CELL 31 — INSPECT FINAL RAG SCORES
# ==========================================================

rag_only = (
    final_df[
        final_df[
            "condition"
        ] == "RAG"
    ]
    .copy()
)


rag_ranked = (
    rag_only[
        [
            "question_id",
            "faithfulness_0_2",
            "clinical_correctness_0_2",
            "completeness_0_2",
            "conciseness_0_1",
            "total_score_0_7",
            "unsupported_claims_yes_no"
        ]
    ]
    .sort_values(
        [
            "total_score_0_7",
            "question_id"
        ],
        ascending=[
            True,
            True
        ]
    )
)


display(
    rag_ranked
)

In [ ]:
# ==========================================================
# CELL 32 — SAVE DEFINITIVE SCORED DATASET
# ==========================================================

FINAL_SCORED_FILE = os.path.join(
    ANALYSIS_DIR,
    "FINAL_HYBRID_unblinded_scored_results.xlsx"
)


if os.path.exists(
    FINAL_SCORED_FILE
):

    print(
        "File already exists."
    )

    print(
        "It will NOT be overwritten:"
    )

    print(
        FINAL_SCORED_FILE
    )


else:

    final_df.to_excel(
        FINAL_SCORED_FILE,
        index=False
    )

    print(
        "✓ Saved:"
    )

    print(
        FINAL_SCORED_FILE
    )

In [ ]:
# ==========================================================
# CELL 33 — SAVE ANALYSIS TABLES
# ==========================================================

ANALYSIS_TABLES_FILE = os.path.join(
    ANALYSIS_DIR,
    "FINAL_HYBRID_stage4_analysis_tables.xlsx"
)


if os.path.exists(
    ANALYSIS_TABLES_FILE
):

    print(
        "Analysis workbook already exists."
    )

    print(
        "It will NOT be overwritten:"
    )

    print(
        ANALYSIS_TABLES_FILE
    )


else:

    with pd.ExcelWriter(
        ANALYSIS_TABLES_FILE,
        engine="openpyxl"
    ) as writer:

        final_df.to_excel(
            writer,
            sheet_name="Final scored data",
            index=False
        )

        descriptive_long.to_excel(
            writer,
            sheet_name="Descriptive statistics",
            index=False
        )

        unsupported_summary.to_excel(
            writer,
            sheet_name="Unsupported claims"
        )

        paired_total.to_excel(
            writer,
            sheet_name="Paired total scores"
        )

        wilcoxon_df.to_excel(
            writer,
            sheet_name="Wilcoxon tests",
            index=False
        )

        mcnemar_table.to_excel(
            writer,
            sheet_name="McNemar table"
        )

        rag_ranked.to_excel(
            writer,
            sheet_name="RAG score ranking",
            index=False
        )


    print(
        "✓ Analysis tables saved:"
    )

    print(
        ANALYSIS_TABLES_FILE
    )

In [ ]:
# ==========================================================
# CELL 34 — FINAL SUMMARY
# ==========================================================

print("=" * 70)

print(
    "FINAL HYBRID STAGE 4 ANALYSIS"
)

print("=" * 70)


print(
    "\nConditions:"
)

print(
    final_df[
        "condition"
    ].value_counts()
)


print(
    "\nUnsupported claims:"
)

display(
    unsupported_summary[
        [
            "unsupported_count",
            "total",
            "percentage"
        ]
    ].round(
        2
    )
)


print(
    "\nPaired total-score comparison:"
)

print(
    "RAG higher:",
    rag_higher
)

print(
    "Tie:",
    ties
)

print(
    "Baseline higher:",
    baseline_higher
)

print(
    "Mean RAG − Baseline difference:",
    round(
        difference.mean(),
        3
    )
)

print(
    "Median RAG − Baseline difference:",
    round(
        difference.median(),
        3
    )
)


print(
    "\nWilcoxon:"
)

display(
    wilcoxon_df[
        [
            "outcome",
            "W",
            "p_value",
            "p_formatted"
        ]
    ]
)


print(
    "\nExact McNemar:"
)

print(
    "p =",
    result.pvalue
)


print(
    "\n✓ Analysis complete"
)